<a href="https://colab.research.google.com/github/jangkj-hongik/hongik-NM2026-1/blob/main/float32_limits_demo_motivation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Floating-Point Limits in Single Precision (`float32`)

This notebook demonstrates several important limitations of floating-point arithmetic using **IEEE-754 single precision** in Python via `numpy.float32`.

It is designed to run directly in **Google Colab**.

Topics covered:
- rounding and loss of tiny terms
- `1 + 10^-24 - 1 = 0`
- absorption and catastrophic cancellation
- non-associativity
- overflow and underflow
- spacing between nearby floating-point numbers


In [ ]:
import numpy as np

np.set_printoptions(precision=30, suppress=False)

def f32(x):
    return np.float32(x)

print('numpy version:', np.__version__)
print('single precision type:', np.float32)


numpy version: 2.0.2
single precision type: <class 'numpy.float32'>


## 1. Tiny numbers can disappear

Mathematically

$$
1 + 10^{-24} - 1 = 10^{-24},
$$

in `float32`, the quantity $10^{-24}$ is far too small to change `1.0`, so:

$$
\mathrm{float32}(1 + 10^{-24}) = 1.
$$

Then subtracting 1 gives 0.

In [ ]:
a = f32(1.0)
tiny = f32(1e-24)
result = f32(a + tiny - a)

print('a         =', a)
print('tiny      =', tiny)
print('a + tiny  =', f32(a + tiny))
print('(a + tiny) - a =', result)


a         = 1.0
tiny      = 1e-24
a + tiny  = 1.0
(a + tiny) - a = 0.0


## 2. Absorption: a small number can be lost when added to a huge number

When numbers differ greatly in magnitude, the smaller one may be completely swallowed.

Mathematically, `(big + small) - big = 1`, but in `float32` it becomes `0` because the `+ 1` is too tiny relative to `10^20`.

In [ ]:
big = f32(1e20)
small = f32(1.0)

print('big              =', big)
print('small            =', small)
print('big + small      =', f32(big + small))
print('(big + small) - big =', f32(f32(big + small) - big))


big              = 1e+20
small            = 1.0
big + small      = 1e+20
(big + small) - big = 0.0


## 3. Catastrophic cancellation

Subtracting nearly equal numbers can destroy significant digits.

A classic example is:

$$
\sqrt{x+1} - \sqrt{x}
$$

for large `x`.

The mathematically equivalent stable form is:

$$
\frac{1}{\sqrt{x+1}+\sqrt{x}}.
$$


In [ ]:
x = f32(1e8)

unstable = f32(np.sqrt(x + f32(1.0)) - np.sqrt(x))
stable = f32(f32(1.0) / (np.sqrt(x + f32(1.0)) + np.sqrt(x)))

print('x =', x)
print('unstable form = sqrt(x+1) - sqrt(x)      =', unstable)
print('stable form   = 1 / (sqrt(x+1)+sqrt(x)) =', stable)


x = 100000000.0
unstable form = sqrt(x+1) - sqrt(x)      = 0.0
stable form   = 1 / (sqrt(x+1)+sqrt(x)) = 5e-05


The unstable version subtracts two very similar quantities, so most meaningful digits cancel.

## 4. Floating-point addition is not associative

In real arithmetic:

$$
(a+b)+c = a+(b+c).
$$

In floating-point arithmetic, this can fail because each intermediate result is rounded.

In [ ]:
a = f32(1e20)
b = f32(-1e20)
c = f32(3.14)

left = f32(f32(a + b) + c)
right = f32(a + f32(b + c))

print('(a + b) + c =', left)
print('a + (b + c) =', right)


(a + b) + c = 3.14
a + (b + c) = 0.0


## 5. Overflow and underflow

Single precision has a largest and smallest representable magnitude.

- Too large  → overflow to `inf`
- Too small  → underflow toward `0`


In [ ]:
info = np.finfo(np.float32)
print('max float32     =', info.max)
print('tiny (min normal) =', info.tiny)
print('smallest subnormal =', np.nextafter(f32(0.0), f32(1.0), dtype=np.float32))

overflow_example = f32(info.max) * f32(2.0)
underflow_example = f32(info.tiny) / f32(2.0)

print('overflow example  =', overflow_example)
print('underflow example =', underflow_example)


max float32     = 3.4028235e+38
tiny (min normal) = 1.1754944e-38
smallest subnormal = 1e-45
overflow example  = inf
underflow example = 5.877472e-39


/tmp/ipykernel_511/703386804.py:6: RuntimeWarning: overflow encountered in scalar multiply
  overflow_example = f32(info.max) * f32(2.0)


## 6. Spacing between nearby representable numbers depends on scale

Floating-point numbers are **not evenly spaced**.

Near larger numbers, the gap between adjacent representable values becomes larger.

In [ ]:
for val in [1.0, 10.0, 1e6, 1e20]:
    x = f32(val)
    next_x = np.nextafter(x, f32(np.inf), dtype=np.float32)
    gap = f32(next_x - x)
    print(f'x = {x: .8e}, nextafter(x, +inf) = {next_x: .8e}, gap = {gap: .8e}')


x =  1.00000000e+00, nextafter(x, +inf) =  1.00000012e+00, gap =  1.19209290e-07
x =  1.00000000e+01, nextafter(x, +inf) =  1.00000010e+01, gap =  9.53674316e-07
x =  1.00000000e+06, nextafter(x, +inf) =  1.00000006e+06, gap =  6.25000000e-02
x =  1.00000002e+20, nextafter(x, +inf) =  1.00000011e+20, gap =  8.79609302e+12


## 7. A small experiment: when does `1 + 10^{-k}` stop changing?

This shows roughly where `float32` can no longer detect the perturbation near 1.

In [ ]:
for k in range(1, 31):
    t = f32(10.0 ** (-k))
    changed = f32(1.0 + t) != f32(1.0)
    print(f'k={k:2d}, 10^(-k)={t:.8e}, changes 1? {changed}')


k= 1, 10^(-k)=1.00000001e-01, changes 1? True
k= 2, 10^(-k)=9.99999978e-03, changes 1? True
k= 3, 10^(-k)=1.00000005e-03, changes 1? True
k= 4, 10^(-k)=9.99999975e-05, changes 1? True
k= 5, 10^(-k)=9.99999975e-06, changes 1? True
k= 6, 10^(-k)=9.99999997e-07, changes 1? True
k= 7, 10^(-k)=1.00000001e-07, changes 1? True
k= 8, 10^(-k)=9.99999994e-09, changes 1? False
k= 9, 10^(-k)=9.99999972e-10, changes 1? False
k=10, 10^(-k)=1.00000001e-10, changes 1? False
k=11, 10^(-k)=9.99999996e-12, changes 1? False
k=12, 10^(-k)=9.99999996e-13, changes 1? False
k=13, 10^(-k)=9.99999982e-14, changes 1? False
k=14, 10^(-k)=9.99999982e-15, changes 1? False
k=15, 10^(-k)=1.00000000e-15, changes 1? False
k=16, 10^(-k)=1.00000002e-16, changes 1? False
k=17, 10^(-k)=9.99999984e-18, changes 1? False
k=18, 10^(-k)=1.00000005e-18, changes 1? False
k=19, 10^(-k)=9.99999968e-20, changes 1? False
k=20, 10^(-k)=9.99999968e-21, changes 1? False
k=21, 10^(-k)=9.99999968e-22, changes 1? False
k=22, 10^(-k)=1.0000

Why does this happen right around k=7?

## Takeaway

Single precision is fast and memory-efficient, but it has strict limits:

1. tiny perturbations can vanish,
2. subtraction of nearly equal numbers can be dangerous,
3. arithmetic identities from real numbers may fail,
4. overflow and underflow are real computational issues,
5. spacing between representable numbers grows with magnitude.

These are exactly the kinds of effects numerical analysis tries to understand and control.